In [21]:
import os
import re
from glob import glob
import pandas as pd

# ========= CONFIG =========
PASTA_BASE = r"C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV"   # raiz com subpastas/arquivos 2010..2023
ANOS = range(2010, 2025)                   # 2010-2023
PADRAO = "**/*.csv"                        # busca recursiva
SEP = ","                                  # CSVs do SIM costumam vir com ','
ENC = "latin-1"                            # comum em dumps do SIM
CHUNKSIZE = 200_000
SAIDA_CSV = r"C:\Users\anami\Downloads\TCC Aninha\Dataset Suicídio.csv"

# Núcleo para epidemiologia
COLS_CORE = [
    "TIPOBITO", "DTOBITO", "HORAOBITO",
    "SEXO", "IDADE", "RACACOR", "ESTCIV", "ESC2010", "ESCFALAGR1", "OCUP",
    "CODMUNRES", "CODMUNOCOR", "LOCOCOR",
    "CAUSABAS", "LINHAA", "LINHAB", "LINHAC", "LINHAD", "LINHAII",
    "CIRCOBITO",
    "ASSISTMED", "NECROPSIA", "FONTE"
]

# Extras úteis para ML/qualidade
COLS_ML_EXTRA = [
    "ALTCAUSA", "DTINVESTIG", "FONTEINV", "TPRESGINFO",
    "DIFDATA", "NUDIASOBCO", "DTCONCASO"
]

COLS_KEEP = list(dict.fromkeys(COLS_CORE + COLS_ML_EXTRA))  # remove duplicatas, preserva ordem

# Regex CAUSABAS X60–X84 (CID-10)
RE_SUIC = re.compile(r"^(X6[0-9]|X7[0-9]|X8[0-4])", re.IGNORECASE)

def listar_arquivos(pasta_base, anos, padrao):
    encontrados = []
    for arq in glob(os.path.join(pasta_base, padrao), recursive=True):
        nome = os.path.basename(arq)
        for ano in anos:
            if str(ano) in arq or str(ano) in nome:
                encontrados.append((ano, arq))
                break
    return sorted(set(encontrados), key=lambda x: (x[0], x[1].lower()))

def decodificar_idade_sim(idade_str):
    """Converte o campo IDADE (SIM) em idade aproximada em anos (float).
       Convenção: 1= minutos, 2= horas, 3= meses, 4= anos, 5= >=100 anos, 9= ignorado."""
    if not isinstance(idade_str, str) or len(idade_str) < 1:
        return None
    unidade = idade_str[0]
    qtd = idade_str[1:3] if len(idade_str) >= 3 else ""
    try:
        q = int(qtd) if qtd else None
    except ValueError:
        q = None

    if unidade == "4":         # anos
        return float(q or 0)
    if unidade == "3":         # meses
        return (q or 0) / 12.0
    if unidade == "2":         # horas
        return (q or 0) / (24*365.0)
    if unidade == "1":         # minutos
        return (q or 0) / (24*60*365.0)
    if unidade == "5":         # >=100 anos (codificação especial)
        return 100.0
    # 9 (ignorado) ou outros -> None
    return None

# Remove arquivo final antigo
if os.path.exists(SAIDA_CSV):
    os.remove(SAIDA_CSV)

arquivos = listar_arquivos(PASTA_BASE, ANOS, PADRAO)
if not arquivos:
    raise SystemExit("Nenhum CSV encontrado (verifique PASTA_BASE/ANOS/PADRAO).")

total_lidas = total_gravadas = 0

for ano, caminho in arquivos:
    print(f"[{ano}] {caminho}")
    for chunk in pd.read_csv(
        caminho, sep=SEP, encoding=ENC, dtype=str, low_memory=False,
        chunksize=CHUNKSIZE, on_bad_lines="skip"
    ):
        # Garante colunas
        for c in COLS_KEEP:
            if c not in chunk.columns:
                chunk[c] = pd.NA

        # Filtrar: não-fetal + CAUSABAS X60–X84
        mask_tipo = chunk["TIPOBITO"].fillna("") != "1"
        mask_cid = chunk["CAUSABAS"].fillna("").str.upper().str.match(RE_SUIC)
        df = chunk.loc[mask_tipo & mask_cid, COLS_KEEP].copy()

        # Derivações úteis p/ análise/ML
        df["ANO_OBITO"] = df["DTOBITO"].str[-4:]             # yyyy
        df["MES_OBITO"] = df["DTOBITO"].str[2:4]             # mm
        df["IDADE_ANOS"] = df["IDADE"].apply(decodificar_idade_sim)

        # Salva (append)
        if not df.empty:
            modo = "a" if os.path.exists(SAIDA_CSV) else "w"
            cab = not os.path.exists(SAIDA_CSV)
            df.to_csv(SAIDA_CSV, sep=",", index=False, encoding="utf-8-sig",
                      mode=modo, header=cab)

        total_lidas += len(chunk)
        total_gravadas += len(df)

print(f"Linhas lidas: {total_lidas:,}")
print(f"Registros X60–X84 gravados: {total_gravadas:,}")
print(f"Arquivo final: {SAIDA_CSV}")


[2010] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2010.csv
[2011] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2011.csv
[2012] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2012.csv
[2013] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2013.csv
[2014] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2014.csv
[2015] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2015.csv
[2016] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2016.csv
[2017] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2017.csv
[2018] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2018.csv
[2019] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2019.csv
[2020] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2020.csv
[2021] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2021.csv
[2022] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2022.csv
[2023] C:\Users\anami\Downloads\TCC Aninha\Arquivos CSV\DOBR2023.csv
[2024] C:\Users\anami\Downloads\TC

In [ ]:
# Configura??o padr?o para figuras vetoriais reprodut?veis
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement ntpath (from versions: none)
ERROR: No matching distribution found for ntpath


In [23]:
tabela = pd.read_csv("Dataset Suicídio.csv")
display(tabela)

C:\Users\anami\AppData\Local\Temp\ipykernel_3636\1079824573.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  tabela = pd.read_csv("Dataset Suicídio.csv")


,TIPOBITO,DTOBITO,HORAOBITO,SEXO,IDADE,RACACOR,ESTCIV,ESC2010,ESCFALAGR1,OCUP,...,ALTCAUSA,DTINVESTIG,FONTEINV,TPRESGINFO,DIFDATA,NUDIASOBCO,DTCONCASO,ANO_OBITO,MES_OBITO,IDADE_ANOS
0,2,31012010,2100,1,410,4.0,1.0,NaN,NaN,999991.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,1,10.0
1,2,7022010,0515,1,424,4.0,1.0,NaN,NaN,999991.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,2,24.0
2,2,19022010,2045,1,414,4.0,1.0,NaN,NaN,999991.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,2,14.0
3,2,20042010,1200,2,426,4.0,2.0,NaN,NaN,999992.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,4,26.0
4,2,24072010,1030,1,428,4.0,2.0,NaN,NaN,999992.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,7,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191020,2,31122024,310.0,1,432,1.0,5.0,1.0,NaN,717020.0,...,NaN,NaN,NaN,NaN,79.0,NaN,NaN,2024,12,32.0
191021,2,25022024,1450.0,1,418,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,138.0,NaN,NaN,2024,2,18.0
191022,2,1032024,1900.0,1,418,5.0,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,133.0,NaN,NaN,2024,3,18.0
191023,2,6052024,1900.0,1,418,5.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,11.0,NaN,NaN,2024,5,18.0


In [25]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# =========================
# CONFIG
# =========================
ARQ_ENTRADA = r"C:\Users\anami\Downloads\TCC Aninha\Dataset Suicídio.csv"
ARQ_SAIDA   = r"C:\Users\anami\Downloads\TCC Aninha\Dataset Suicídio 2010-2024.csv"
SEP_IN      = ","        # seu consolidado foi salvo com ';'
ENC_IN      = "utf-8-sig"

# =========================
# HELPERS
# =========================
def to_datetime_ddmmyyyy(series):
    """Converte datas ddmmaaaa para datetime (coerção segura)."""
    # Preenche com NaN se vier vazio/curto
    s = series.fillna("").astype(str).str.strip()
    s = s.where(s.str.len() == 8, np.nan)
    return pd.to_datetime(s, format="%d%m%Y", errors="coerce")

def decode_idade_sim(idade_str):
    """
    Converte IDADE (SIM) para anos (float).
    Convenção do SIM:
      1=minutos, 2=horas, 3=meses, 4=anos, 5= >=100 anos, 9=ignorado.
    """
    if not isinstance(idade_str, str) or len(idade_str) < 1:
        return np.nan
    unidade = idade_str[0]
    qtd = idade_str[1:3] if len(idade_str) >= 3 else ""
    try:
        q = int(qtd) if qtd else None
    except ValueError:
        q = None

    if unidade == "4":         # anos
        return float(q or 0)
    if unidade == "3":         # meses
        return (q or 0) / 12.0
    if unidade == "2":         # horas
        return (q or 0) / (24*365.0)
    if unidade == "1":         # minutos
        return (q or 0) / (24*60*365.0)
    if unidade == "5":         # >= 100 anos (categoria especial)
        return 100.0
    return np.nan  # 9=ignorado ou inválidos

def estacao_brasil(dt: pd.Timestamp) -> str:
    """Retorna estação (Brasil, hemisfério sul) a partir da data."""
    # Datas aproximadas (solstícios/equinócios podem variar +/- 1 dia)
    y = dt.year
    inicio_out = datetime(y, 3, 20)
    inicio_inv = datetime(y, 6, 21)
    inicio_prim = datetime(y, 9, 23)
    inicio_ver = datetime(y, 12, 21)
    if dt >= inicio_ver or dt < inicio_out:
        return "Verao"
    elif dt >= inicio_out and dt < inicio_inv:
        return "Outono"
    elif dt >= inicio_inv and dt < inicio_prim:
        return "Inverno"
    else:
        return "Primavera"

def normaliza_cod_mun(cod):
    """Garante 7 dígitos (IBGE município)."""
    s = ("" if pd.isna(cod) else str(cod)).strip()
    s = re.sub(r"\D", "", s)  # remove não numérico, se houver
    return s.zfill(7) if s else np.nan

def uf_from_codmun(cod7):
    """Extrai UF (2 dígitos) do código IBGE de 7 dígitos."""
    if not isinstance(cod7, str) or len(cod7) < 2:
        return np.nan
    return cod7[:2]

def any_cid_in_range(texto: str, prefixo: str, ini: int, fim: int) -> bool:
    """
    Verifica se há algum CID no texto que pertença ao intervalo prefixo+ini..prefixo+fim.
    Ex.: F20-F99 => prefixo='F', ini=20, fim=99
    """
    if not isinstance(texto, str):
        return False
    # captura tokens tipo F20, F31, etc.
    cids = re.findall(r"\b([A-Z]\d{2})\b", texto.upper())
    for cid in cids:
        if cid.startswith(prefixo):
            try:
                num = int(cid[1:3])
                if ini <= num <= fim:
                    return True
            except ValueError:
                continue
    return False

def limpa_ignorado(col: pd.Series, valores_ign="9 99 999 9999 0 I".split()):
    """Padroniza códigos de 'ignorado' em NaN (para algumas colunas categóricas)."""
    s = col.astype(str).str.strip()
    return s.mask(s.isin(valores_ign) | s.eq("") | s.eq("nan"))

# Mapeamentos de categorias (SIM)
MAP_SEXO = {
    "1": "Masculino", "M": "Masculino",
    "2": "Feminino",  "F": "Feminino",
    "0": np.nan, "9": np.nan, "I": np.nan
}
MAP_RACACOR = {
    "1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indigena", "9": np.nan
}
MAP_ESTCIV = {
    "1": "Solteiro", "2": "Casado", "3": "Viuvo", "4": "Separado/Divorciado",
    "5": "Uniao_estavel", "9": np.nan
}
MAP_ESC2010 = {
    "0": "Sem_escolaridade",
    "1": "Fundamental_I",
    "2": "Fundamental_II",
    "3": "Medio",
    "4": "Superior_incompleto",
    "5": "Superior_completo",
    "9": np.nan
}
MAP_LOCOCOR = {
    "1": "Hospital",
    "2": "Outro_estab_saude",
    "3": "Domicilio",
    "4": "Via_publica",
    "5": "Outros",
    "6": "Aldeia_indigena",
    "9": np.nan
}

def classifica_meio_suicidio(causabas: str) -> str:
    """
    Agrupamento amplo do meio a partir da CAUSABAS (X60–X84):
      X60–X69: intoxicação/autoenvenenamento
      X70–X79: meios mecânicos (enforcamento, afogamento, arma de fogo etc.)
      X80–X84: salto/outros meios especificados
    """
    if not isinstance(causabas, str):
        return np.nan
    c = causabas.upper().strip()
    if re.match(r"^X6[0-9]", c):
        return "Intoxicacao_autoenvenenamento"
    if re.match(r"^X7[0-9]", c):
        return "Meios_mecanicos"
    if re.match(r"^X8[0-4]", c):
        return "Salto_e_outros"
    return np.nan

# =========================
# CARGA
# =========================
df = pd.read_csv(ARQ_ENTRADA, sep=SEP_IN, encoding=ENC_IN, dtype=str, low_memory=False)

# =========================
# LIMPEZA & PADRONIZAÇÃO
# =========================

# Garante não-fetal (por precaução; seu consolidado já deve estar filtrado)
if "TIPOBITO" in df.columns:
    df = df[df["TIPOBITO"].fillna("") != "1"].copy()  # 1 = fetal

# Datas e derivados
df["DATA_OBITO"] = to_datetime_ddmmyyyy(df.get("DTOBITO"))
df["ANO_OBITO"]  = df["DATA_OBITO"].dt.year
df["MES_OBITO"]  = df["DATA_OBITO"].dt.month
df["DIA_OBITO"]  = df["DATA_OBITO"].dt.day
df["SEMANA_OBITO"] = df["DATA_OBITO"].dt.isocalendar().week.astype("Int64")
df["DIA_DA_SEMANA"] = df["DATA_OBITO"].dt.day_name(locale="pt_BR").fillna(df["DATA_OBITO"].dt.day_name())

# Estação (Brasil)
df["ESTACAO"] = df["DATA_OBITO"].apply(lambda d: estacao_brasil(d) if pd.notna(d) else np.nan)

# Idade (anos) a partir de IDADE
df["IDADE_ANOS"] = df["IDADE"].apply(decode_idade_sim)

# Faixas etárias
bins = [-np.inf, 14, 29, 44, 59, np.inf]
labs = ["<15", "15-29", "30-44", "45-59", "60+"]
df["FAIXA_ETARIA"] = pd.cut(df["IDADE_ANOS"], bins=bins, labels=labs, right=True)

# Códigos de município (7 dígitos) e UFs
df["CODMUNRES"]  = df["CODMUNRES"].apply(normaliza_cod_mun) if "CODMUNRES" in df.columns else np.nan
df["CODMUNOCOR"] = df["CODMUNOCOR"].apply(normaliza_cod_mun) if "CODMUNOCOR" in df.columns else np.nan
df["UF_RES"]     = df["CODMUNRES"].apply(uf_from_codmun)
df["UF_OCOR"]    = df["CODMUNOCOR"].apply(uf_from_codmun)

# Mapeia categóricas
df["SEXO_CAT"]     = df["SEXO"].map(MAP_SEXO)
df["RACACOR_CAT"]  = df["RACACOR"].map(MAP_RACACOR)
df["ESTCIV_CAT"]   = df["ESTCIV"].map(MAP_ESTCIV)
df["ESC2010_CAT"]  = df["ESC2010"].map(MAP_ESC2010) if "ESC2010" in df.columns else np.nan
df["LOCOCOR_CAT"]  = df["LOCOCOR"].map(MAP_LOCOCOR)

# Padroniza campos com 'ignorado' (opcional, já mapeamos alguns acima)
for col in ["SEXO_CAT", "RACACOR_CAT", "ESTCIV_CAT", "ESC2010_CAT", "LOCOCOR_CAT"]:
    df[col] = df[col].replace({"nan": np.nan})

# =========================
# FEATURE ENGINEERING
# =========================

# Meio (a partir de CAUSABAS X60–X84)
df["MEIO_SUICIDIO"] = df["CAUSABAS"].apply(classifica_meio_suicidio)

# Contagem de causas informadas (LINHAA..LINHAD, LINHAII)
lin_cols = [c for c in ["LINHAA", "LINHAB", "LINHAC", "LINHAD", "LINHAII"] if c in df.columns]
df["NUM_CAUSAS_INFORMADAS"] = df[lin_cols].notna().sum(axis=1) if lin_cols else 0

# Indicadores de comorbidades nas linhas (CID-10)
def concat_causas(row):
    partes = []
    for c in lin_cols:
        v = row.get(c)
        if isinstance(v, str):
            partes.append(v)
    return " ".join(partes)

if lin_cols:
    causas_txt = df.apply(concat_causas, axis=1)
    df["FLAG_TP_MENTAL_F20_F99"] = causas_txt.apply(lambda t: any_cid_in_range(t, "F", 20, 99)).astype(int)
    df["FLAG_TP_SUBST_F10_F19"]  = causas_txt.apply(lambda t: any_cid_in_range(t, "F", 10, 19)).astype(int)
else:
    df["FLAG_TP_MENTAL_F20_F99"] = 0
    df["FLAG_TP_SUBST_F10_F19"]  = 0

# Circunstância do óbito (CIRCOBITO): manter para auditoria (2=suicídio)
# Se desejar, já criar uma flag de consistência:
if "CIRCOBITO" in df.columns:
    df["FLAG_CIRC_SUIC"] = (df["CIRCOBITO"].astype(str).str.strip() == "2").astype(int)

# =========================
# COLUNAS FINAIS (sugestão para EDA)
# =========================
COLS_FINAIS = [
    # chaves/tempo/local
    "DATA_OBITO","ANO_OBITO","MES_OBITO","DIA_OBITO","SEMANA_OBITO","DIA_DA_SEMANA","ESTACAO",
    "CODMUNRES","UF_RES","CODMUNOCOR","UF_OCOR","LOCOCOR","LOCOCOR_CAT",
    # demografia/ocupação/escolaridade
    "SEXO","SEXO_CAT","IDADE","IDADE_ANOS","FAIXA_ETARIA","RACACOR","RACACOR_CAT",
    "ESTCIV","ESTCIV_CAT","ESC2010","ESC2010_CAT","OCUP",
    # causa/circunstância
    "CAUSABAS","MEIO_SUICIDIO","CIRCOBITO","FLAG_CIRC_SUIC",
    "LINHAA","LINHAB","LINHAC","LINHAD","LINHAII","NUM_CAUSAS_INFORMADAS",
    # comorbidades proxy
    "FLAG_TP_MENTAL_F20_F99","FLAG_TP_SUBST_F10_F19",
    # campos originais úteis para rastreio
    "ASSISTMED","NECROPSIA","FONTE","TIPOBITO"
]
COLS_FINAIS = [c for c in COLS_FINAIS if c in df.columns]  # garante existência

# =========================
# SALVA
# =========================
df_final = df[COLS_FINAIS].copy()
df_final.to_csv(ARQ_SAIDA, sep=",", index=False, encoding="utf-8-sig")
print(f"OK! Base pronta para EDA: {ARQ_SAIDA}")
print(f"Linhas: {len(df_final):,} | Colunas: {len(df_final.columns)}")


OK! Base pronta para EDA: C:\Users\anami\Downloads\TCC Aninha\Dataset Suicídio 2010-2024.csv
Linhas: 191,025 | Colunas: 41


In [27]:
tabela = pd.read_csv("Dataset Suicídio 2010-2024.csv")
display(tabela)

C:\Users\anami\AppData\Local\Temp\ipykernel_3636\1922078845.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  tabela = pd.read_csv("Dataset Suicídio 2010-2024.csv")


,DATA_OBITO,ANO_OBITO,MES_OBITO,DIA_OBITO,SEMANA_OBITO,DIA_DA_SEMANA,ESTACAO,CODMUNRES,UF_RES,CODMUNOCOR,...,LINHAC,LINHAD,LINHAII,NUM_CAUSAS_INFORMADAS,FLAG_TP_MENTAL_F20_F99,FLAG_TP_SUBST_F10_F19,ASSISTMED,NECROPSIA,FONTE,TIPOBITO
0,2010-01-31,2010,1,31,4,Domingo,Verao,120060,1,120060,...,*X780,NaN,NaN,3,0,0,NaN,2.0,3.0,2
1,2010-02-07,2010,2,7,5,Domingo,Verao,120060,1,120060,...,NaN,NaN,NaN,1,0,0,NaN,2.0,3.0,2
2,2010-02-19,2010,2,19,7,Sexta-feira,Verao,120060,1,120060,...,NaN,NaN,NaN,2,0,0,NaN,2.0,3.0,2
3,2010-04-20,2010,4,20,16,Terça-feira,Outono,120060,1,120060,...,*X780,NaN,NaN,3,0,0,NaN,2.0,3.0,2
4,2010-07-24,2010,7,24,29,Sábado,Inverno,120060,1,120060,...,NaN,NaN,NaN,1,0,0,NaN,2.0,3.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191020,2024-12-31,2024,12,31,1,Terça-feira,Verao,310160,3,310160,...,NaN,NaN,NaN,2,0,0,2.0,1.0,1.0,2
191021,2024-02-25,2024,2,25,8,Domingo,Verao,130140,1,130140,...,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN,2
191022,2024-03-01,2024,3,1,9,Sexta-feira,Verao,130140,1,130140,...,NaN,NaN,NaN,1,0,0,NaN,NaN,NaN,2
191023,2024-05-06,2024,5,6,19,Segunda-feira,Outono,150510,1,150510,...,NaN,NaN,NaN,2,0,0,NaN,NaN,NaN,2
